## Sentiment Analysis with https://huggingface.co/siebert/sentiment-roberta-large-english derived from the paper: https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0276367  

In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
labels = pd.read_csv('/content/drive/MyDrive/LSE/Capstone/data/final_topic_labels.csv')

In [ ]:
data =  pd.read_csv('/content/drive/MyDrive/LSE/Capstone/data/meta_final.csv')

In [ ]:
data = data.merge(labels[['Topic', 'Name']], how='left', left_on='topic', right_on='Topic')

In [ ]:
data_filtered = data[data['topic'].isin([0, 1, 2, 3, 4])]

In [ ]:
from transformers import pipeline
sentiment_analysis = pipeline("sentiment-analysis",model="siebert/sentiment-roberta-large-english")
print(sentiment_analysis("I love this!"))

In [ ]:
import pandas as pd
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
from datasets import Dataset
from tqdm import tqdm

clf = pipeline(
    "sentiment-analysis",
    model="siebert/sentiment-roberta-large-english", 
    device = 0,         # -1 for CPU, "mps" on Apple silicon
    truncation=True,
    max_length=128, 
    top_k=None
)

ds = Dataset.from_pandas(data_filtered[["sentence"]])
results = [r for r in tqdm(clf(KeyDataset(ds, "sentence"), batch_size=32), total=len(ds))]

data_filtered["p_pos"] = [
    next(x["score"] for x in r if x["label"].upper() == "POSITIVE") for r in results
]
data_filtered["p_neg"] = 1 - data_filtered["p_pos"]

In [ ]:
data_filtered["sentiment_label"] = [r["label"] for r in results]
data_filtered["sentiment_probability"] = [r["score"] for r in results]
data_filtered["p_pos"] = [r["score"] if r["label"] == "POSITIVE" else 1 - r["score"] for r in results]
data_filtered["p_neg"] = 1 - data_filtered["p_pos"]

In [ ]:
data_filtered.to_csv("/content/drive/MyDrive/LSE/Capstone/data/sentimentsOut.csv", index=False)